## Table 2: ViT-Base model's class leakage results on GTSRB dataset using RTX 4500 Ada GPU

In [1]:
import numpy as np
import pandas as pd

### Section 1: Loading Dataset

In [2]:
all_class_data=[]
for i in range(1,101):
    temp=[]
    for j in range(43):
        path='/home/rtx3060/Desktop/GPU_Trace/A4500_test/mem_csv_vit_base_gtsrb_class/report'+str(j)+'_'+str(i)+'.csv'
        data=pd.read_csv(path)
        temp.append(data)
    all_class_data.append(temp)
    
print(all_class_data[0][1].columns)

Index(['ID', 'Process ID', 'Process Name', 'Host Name', 'Kernel Name',
       'Context', 'Stream', 'Block Size', 'Grid Size', 'Device',
       ...
       'smsp__warps_active.min.peak_sustained',
       'smsp__warps_active.min.per_cycle_active',
       'smsp__warps_active.sum.peak_sustained',
       'smsp__warps_active.sum.per_cycle_active',
       'smsp__warps_eligible.avg.per_cycle_active',
       'smsp__warps_eligible.max.per_cycle_active',
       'smsp__warps_eligible.min.per_cycle_active',
       'smsp__warps_eligible.sum.per_cycle_active', 'thread_inst_executed',
       'thread_inst_executed_true'],
      dtype='object', length=1220)


### Section 2: Pre-processing Dataset

In [3]:
for i in range(100):
    for j in range(43):
        all_class_data[i][j]=all_class_data[i][j].drop(['ID'], axis=1)
        if 0 in all_class_data[i][j].index:
            all_class_data[i][j] = all_class_data[i][j].drop(index=0)

In [4]:
all_class_data[0][0]['Kernel Name'].unique

<bound method Series.unique of 1     fmha_cutlassF_f32_aligned_64x64_rf_sm80(Attent...
2     fmha_cutlassF_f32_aligned_64x64_rf_sm80(Attent...
3     fmha_cutlassF_f32_aligned_64x64_rf_sm80(Attent...
4     fmha_cutlassF_f32_aligned_64x64_rf_sm80(Attent...
5     fmha_cutlassF_f32_aligned_64x64_rf_sm80(Attent...
6     fmha_cutlassF_f32_aligned_64x64_rf_sm80(Attent...
7     fmha_cutlassF_f32_aligned_64x64_rf_sm80(Attent...
8     fmha_cutlassF_f32_aligned_64x64_rf_sm80(Attent...
9     fmha_cutlassF_f32_aligned_64x64_rf_sm80(Attent...
10    fmha_cutlassF_f32_aligned_64x64_rf_sm80(Attent...
11    fmha_cutlassF_f32_aligned_64x64_rf_sm80(Attent...
12    fmha_cutlassF_f32_aligned_64x64_rf_sm80(Attent...
Name: Kernel Name, dtype: object>

### Section 3: Metric-based Data filtering and pre-processing

In [5]:
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib

classes=[i for i in range(43)] 
kernel_name=['fmha_cutlassF_f32_aligned_64x64_rf_sm80']
ltx_cols = [col for col in all_class_data[0][1].columns if col.startswith('thread_inst_executed')]# or col.startswith('gpu__') or col.startswith('sm__')]
for col in ltx_cols:
    for i in range(100):
        for j in range(43):
            if all_class_data[i][j][col].dtype == 'object':  # likely string
                all_class_data[i][j][col] = pd.to_numeric(all_class_data[i][j][col].astype(str).str.replace(',', '').str.strip(), errors='coerce')

### Section 4: Generating classifier model's training data and validation data by considering only the filtered Metrics.

In [6]:
col = [col for col in all_class_data[0][1].columns if 
       col=='thread_inst_executed'  
        or col.startswith('smsp__branch_targets_threads_divergent') #
        or col.startswith('smsp__inst_executed_op_shared_atom') #
        or col.startswith('smsp__sass_branch_targets_thread_divergent') #
]

for i in range(100):
    for j in range(43):
        for c in col:
            all_class_data[i][j][c]=pd.to_numeric(all_class_data[i][j][c].astype(str).str.replace(',', '').str.strip(), errors='coerce').astype('float32')

# Loop through each DataFrame and keep only the selected columns (if they exist)
df=[[[] for i in range(43)] for j in range(100)]
for i in range(len(all_class_data)):
    for j in range(len(all_class_data[i])):
        df2 = all_class_data[i][j]
        # Keep only the columns that exist in the current DataFrame
        selected_cols = [c for c in col if c in df2.columns]
        df[i][j] = df2[selected_cols].copy()

In [7]:
df[0][0].columns

Index(['smsp__branch_targets_threads_divergent',
       'smsp__inst_executed_op_shared_atom.sum',
       'smsp__inst_executed_op_shared_atom.sum.pct_of_peak_sustained_elapsed',
       'thread_inst_executed'],
      dtype='object')

### Dataset re-arrangement

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X=[]
y=[]
for i in range(100):
    for j in range(43):
        # t=scaler.fit_transform(df[i][j])
        t=df[i][j]
        X.append(t)
        y.append(j)
X = np.array(X)
y = np.array(y)
# X = X[:, 10:11, :]
mean = X.mean(axis=(0, 1), keepdims=True)
std = X.std(axis=(0, 1), keepdims=True)
X = (X - mean) / (std + 1e-8)
print(X.shape, y.shape)

(4300, 12, 4) (4300,)


In [9]:
import torch
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)
inp_ch= X.shape[2]

cuda:0


### Attack Classifier training and 5-fold cross validation

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CNNTimeSeriesClassifier(nn.Module):
    def __init__(self, num_classes=43):
        super(CNNTimeSeriesClassifier, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=inp_ch, out_channels=32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(32)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(64)
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        # Input shape: (batch_size, 12, 6)
        x = x.permute(0, 2, 1)  # Convert to (batch_size, 6, 12)
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)  # (batch_size, 64, 1)
        x = x.squeeze(-1)  # (batch_size, 64)
        x = self.fc(x)  # (batch_size, num_classes)
        return x


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import seaborn as sns
from torch.optim.lr_scheduler import ReduceLROnPlateau



X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, y_train.shape)

batch_size = 60
train_dataset = torch.utils.data.TensorDataset(X_train, y_train)
val_dataset = torch.utils.data.TensorDataset(X_val, y_val)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size)

torch.Size([3440, 12, 4]) torch.Size([3440])


In [12]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import random

random.seed(10)

# Configuration
k_folds = 5
num_epochs = 100
batch_size = 60
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)

fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n--- Fold {fold} ---")

    # Create data for current fold
    X_train_fold = X[train_idx].clone().detach().float()
    y_train_fold = y[train_idx].clone().detach().long()
    X_val_fold = X[val_idx].clone().detach().float()
    y_val_fold = y[val_idx].clone().detach().long()

    train_dataset = TensorDataset(X_train_fold, y_train_fold)
    val_dataset = TensorDataset(X_val_fold, y_val_fold)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    # New model instance per fold
    model = CNNTimeSeriesClassifier().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)#, verbose=True)

    for epoch in range(1, num_epochs + 1):
        model.train()
        total_loss = 0
        y_true_train, y_pred_train = [], []

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            y_pred_train.extend(torch.argmax(out, dim=1).cpu().numpy())
            y_true_train.extend(yb.cpu().numpy())

        train_acc = accuracy_score(y_true_train, y_pred_train)

        # Validation
        model.eval()
        y_true_val, y_pred_val = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                out = model(xb)
                y_pred_val.extend(torch.argmax(out, dim=1).cpu().numpy())
                y_true_val.extend(yb.cpu().numpy())

        val_acc = accuracy_score(y_true_val, y_pred_val)
        
        print(f"Fold {fold} | Epoch {epoch:02d} | Loss: {total_loss:.4f} "
              f"| Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}%")

    # Store final accuracy for this fold
    fold_results.append(val_acc)

# Summary
print("\n=== K-Fold Cross Validation Results ===")
print(f"Accuracies for each fold: {[f'{acc*100:.2f}%' for acc in fold_results]}")
print(f"Average Accuracy: {np.mean(fold_results)*100:.2f}%")



--- Fold 1 ---
Fold 1 | Epoch 01 | Loss: 220.3470 | Train Acc: 7.97% | Val Acc: 11.74%
Fold 1 | Epoch 02 | Loss: 199.5560 | Train Acc: 15.55% | Val Acc: 17.56%
Fold 1 | Epoch 03 | Loss: 185.4803 | Train Acc: 22.01% | Val Acc: 24.07%
Fold 1 | Epoch 04 | Loss: 171.7616 | Train Acc: 28.98% | Val Acc: 29.88%
Fold 1 | Epoch 05 | Loss: 158.7051 | Train Acc: 35.38% | Val Acc: 34.53%
Fold 1 | Epoch 06 | Loss: 147.0407 | Train Acc: 38.98% | Val Acc: 39.07%
Fold 1 | Epoch 07 | Loss: 135.1678 | Train Acc: 45.78% | Val Acc: 47.67%
Fold 1 | Epoch 08 | Loss: 124.1420 | Train Acc: 50.96% | Val Acc: 51.86%
Fold 1 | Epoch 09 | Loss: 114.3555 | Train Acc: 55.76% | Val Acc: 56.40%
Fold 1 | Epoch 10 | Loss: 104.6358 | Train Acc: 62.03% | Val Acc: 58.60%
Fold 1 | Epoch 11 | Loss: 96.2159 | Train Acc: 65.35% | Val Acc: 65.58%
Fold 1 | Epoch 12 | Loss: 88.2742 | Train Acc: 70.29% | Val Acc: 68.84%
Fold 1 | Epoch 13 | Loss: 80.8130 | Train Acc: 74.04% | Val Acc: 73.26%
Fold 1 | Epoch 14 | Loss: 74.0506 | Tra